In [1]:
import cantera as ct

# --- Define your solutions ---
T_chamber = 1200  # K, preheated to avoid stiff-chem crash
P_chamber = 3.45e6  # Pa
T_amb = 300
P_amb = 1e5

# RP-1 mechanism
kerosene = ct.Solution("A2NTC_skeletal.yaml")
lox      = ct.Solution("A2NTC_skeletal.yaml")
exhaust  = ct.Solution("A2NTC_skeletal.yaml")

# Mass fractions
X_kero = "POSF10325:1"  # example, adjust to mechanism
X_lox  = "O2:1"
X_exh  = "O2:0.21, N2:0.79"

# Set initial states
kerosene.TPX = T_chamber, P_chamber, X_kero
lox.TPX     = T_chamber, P_chamber, X_lox
exhaust.TP  = T_amb, P_amb

# Mass flow rates (kg/s)
mdot_lox  = 0.1
mdot_kero = 0.05

# --- Reservoirs ---
res_lox  = ct.Reservoir(lox, name="LOX Reservoir")
res_kero = ct.Reservoir(kerosene, name="Kerosene Reservoir")
outlet   = ct.Reservoir(exhaust, name="Outlet")

# --- Reactor ---
chamber = ct.ConstPressureReactor(kerosene, name="Chamber", energy='on', clone=True)

# --- Mass flow controllers into chamber ---
mfc_lox  = ct.MassFlowController(res_lox, chamber, mdot=mdot_lox, name="LOX Inlet")
mfc_kero = ct.MassFlowController(res_kero, chamber, mdot=mdot_kero, name="Kero Inlet")

# --- Pressure controller at outlet ---
pc_outlet = ct.PressureController(chamber, outlet, primary=mfc_lox, K=1e-5)

# --- Reactor network ---
sim = ct.ReactorNet([chamber])

# --- Time stepping example ---
t = 0.0
dt = 1e-6
max_time = 0.01

times = []
temps = []
X_fuel = []

while t < max_time:
    t = sim.step()
    times.append(t)
    temps.append(chamber.T)

# --- Example output ---
for ti, T, X in zip(times, temps, X_fuel):
    print(f"t={ti:.6f} s, T={T:.1f} K, RP-1 X={X:.4f}")
temps

CanteraError: 
*******************************************************************************
CanteraError thrown by Application::findInputFile:

Input file A2NTC_skeletal.yaml not found in directories 
'.', 
'/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/cantera/data'

To fix this problem, either:
    a) move the missing files into the local directory;
    b) define environment variable CANTERA_DATA to
         point to the directory containing the file.
*******************************************************************************
